# **1. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import GridSearchCV


# **2. Memuat Dataset dari Hasil Clustering**

Memuat dataset hasil clustering dari file CSV ke dalam variabel DataFrame.

In [2]:
df = pd.read_csv('clusters_dataset.csv')
df.head()

,age,bmi,blood_glucose_level,hbA1c_level,cluster
0,32.0,27.32,100.0,5.0,1
1,29.0,19.95,90.0,5.0,0
2,18.0,23.76,160.0,4.8,0
3,41.0,27.32,159.0,4.0,1
4,52.0,23.75,90.0,6.5,2


# **3. Data Splitting**

Tahap Data Splitting bertujuan untuk memisahkan dataset menjadi dua bagian: data latih (training set) dan data uji (test set).

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# **4. Membangun Model Klasifikasi**


## **a. Membangun Model Klasifikasi**

Setelah memilih algoritma klasifikasi yang sesuai, langkah selanjutnya adalah melatih model menggunakan data latih.

Berikut adalah rekomendasi tahapannya.
1. Pilih algoritma klasifikasi yang sesuai, seperti Logistic Regression, Decision Tree, Random Forest, atau K-Nearest Neighbors (KNN).
2. Latih model menggunakan data latih.

## Metode 1 (Random Forest)
Random Forest adalah algoritma klasifikasi berbasis ensemble learning yang menggunakan banyak decision tree untuk membuat prediksi.
# Cara Kerja:
- Dataset diacak dan dibagi menjadi beberapa subset dengan bootstrapping.
- Setiap decision tree dilatih dengan subset data yang berbeda.
- Saat klasifikasi, setiap tree memberikan vote untuk suatu kelas.
- Kelas dengan vote terbanyak menjadi hasil akhir (majority voting).

In [4]:
# Pisahkan fitur dan label
X_train = train_df.drop(columns=['cluster'])
y_train = train_df['cluster']

# Inisialisasi dan latih model Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## Metode 2 K-Nearest Neighbors (KNN)
KNN adalah algoritma berbasis instance-based learning yang mengklasifikasikan data berdasarkan tetangga terdekat dalam ruang fitur.
# Cara Kerja:
- Tentukan nilai K (jumlah tetangga terdekat).
- Hitung jarak antara data baru dengan semua data dalam dataset (biasanya dengan Euclidean distance).
- Pilih K data terdekat.
- Kelas yang paling banyak muncul di antara K tetangga akan menjadi kelas untuk data baru (majority voting).

Saya ingin coba melakukan Hyperparameter Tuning dengan metode grid search sebelum memilih n_neighbors

In [6]:
param_grid = {'n_neighbors': [3, 5, 7, 9, 11]}
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)

{'n_neighbors': 3}


Terlihat hasil terbaik ada pada n_neighbors = 3 jadi saya gunakan ini.

In [9]:
# Inisialisasi dan latih model KNN
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=3)

Tulis narasi atau penjelasan algoritma yang Anda gunakan.

## **b. Evaluasi Model Klasifikasi**

Berikut adalah **rekomendasi** tahapannya.
1. Lakukan prediksi menggunakan data uji.
2. Hitung metrik evaluasi seperti Accuracy dan F1-Score (Opsional: Precision dan Recall).
3. Buat confusion matrix untuk melihat detail prediksi benar dan salah.

## Metode 1 (Random Forest)

In [12]:
# Pisahkan fitur dan label untuk data uji
X_test = test_df.drop(columns=['cluster'])
y_test = test_df['cluster']

# Prediksi dengan Random Forest
y_pred_rf = rf_model.predict(X_test)

# Hitung metrik evaluasi
accuracy_rf = accuracy_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf, average='weighted')
conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)

# Tampilkan hasil
print("Random Forest - Accuracy:", accuracy_rf)
print("Random Forest - F1 Score:", f1_rf)
print("Random Forest - Confusion Matrix:\n", conf_matrix_rf)

Random Forest - Accuracy: 0.9899020996244073
Random Forest - F1 Score: 0.989903197442804
Random Forest - Confusion Matrix:
 [[4954   24   37]
 [  26 4449   14]
 [  42   21 6674]]


## Metode 2 (KNN)

In [15]:
# Prediksi dengan KNN
y_pred_knn = knn_model.predict(X_test)

# Hitung metrik evaluasi
accuracy_knn = accuracy_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
conf_matrix_knn = confusion_matrix(y_test, y_pred_knn)

# Tampilkan hasil
print("KNN - Accuracy:", accuracy_knn)
print("KNN - F1 Score:", f1_knn)
print("KNN - Confusion Matrix:\n", conf_matrix_knn)


KNN - Accuracy: 0.9427990887260637
KNN - F1 Score: 0.9423937721594955
KNN - Confusion Matrix:
 [[4884   68   63]
 [ 177 3951  361]
 [  77  183 6477]]


- Dapat dilihat 2 jenis model yang dihasilkan dari 2 algoitma yang berbeda. Keduanya memiliki nilai akurasi dan F1-Score yang sangat baik yaitu di atas 92%. 
- Terlihat model random forest lebih baik untuk jenis dataset ini ketimbang KNN, bahkan akurasi dan F1-Score nya mencapai 98%

Tulis hasil evaluasi algoritma yang digunakan, jika Anda menggunakan 2 algoritma, maka bandingkan hasilnya.

## **c. Tuning Model Klasifikasi (Optional)**

Gunakan GridSearchCV, RandomizedSearchCV, atau metode lainnya untuk mencari kombinasi hyperparameter terbaik

## **d. Evaluasi Model Klasifikasi setelah Tuning (Optional)**

Berikut adalah rekomendasi tahapannya.
1. Gunakan model dengan hyperparameter terbaik.
2. Hitung ulang metrik evaluasi untuk melihat apakah ada peningkatan performa.

Saya sudah coba tuning di atas, tapi hanya untuk algoritma kedua, dan saya juga hanya mengambil nilai terbaik berdasarkan algoritma grid search tanpa membandingkan hasil dari masing-masingnya.

## **e. Analisis Hasil Evaluasi Model Klasifikasi**

Berikut adalah **rekomendasi** tahapannya.
1. Bandingkan hasil evaluasi sebelum dan setelah tuning (jika dilakukan).
2. Identifikasi kelemahan model, seperti:
  - Precision atau Recall rendah untuk kelas tertentu.
  - Apakah model mengalami overfitting atau underfitting?
3. Berikan rekomendasi tindakan lanjutan, seperti mengumpulkan data tambahan atau mencoba algoritma lain jika hasil belum memuaskan.

In [26]:
# Evaluasi Random Forest
train_acc_rf = accuracy_score(y_train, rf_model.predict(X_train))
test_acc_rf = accuracy_score(y_test, rf_model.predict(X_test))

print("Random Forest - Training Accuracy:", train_acc_rf)
print("Random Forest - Testing Accuracy:", test_acc_rf)

# Evaluasi KNN
train_acc_knn = accuracy_score(y_train, knn_model.predict(X_train))
test_acc_knn = accuracy_score(y_test, knn_model.predict(X_test))

print("KNN - Training Accuracy:", train_acc_knn)
print("KNN - Testing Accuracy:", test_acc_knn)

Random Forest - Training Accuracy: 1.0
Random Forest - Testing Accuracy: 0.9899020996244073
KNN - Training Accuracy: 0.9775246305418719
KNN - Testing Accuracy: 0.9427990887260637


In [25]:
# Evaluasi Random Forest
precision_rf = precision_score(y_test, rf_model.predict(X_test), average='weighted')
recall_rf = recall_score(y_test, rf_model.predict(X_test), average='weighted')

print("Random Forest - Precision:", precision_rf)
print("Random Forest - Recall:", recall_rf)

# Evaluasi KNN
precision_knn = precision_score(y_test, knn_model.predict(X_test), average='weighted')
recall_knn = recall_score(y_test, knn_model.predict(X_test), average='weighted')

print("KNN - Precision:", precision_knn)
print("KNN - Recall:", recall_knn)

Random Forest - Precision: 0.9899054150602302
Random Forest - Recall: 0.9899020996244073
KNN - Precision: 0.9427383346988144
KNN - Recall: 0.9427990887260637


- Precision dan recall bagus dan sepertinya tidak terjadi overfitting atau underfitting karena training dan testing akurasi sama-sama bagus.
- Sebenarnya saya tidak ada saran lanjutan karena hasilnya memuaskan hehe.